In [1]:
! pip install librosa soundfile requests

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [2]:
! pip install librosa soundfile requests

Active code page: 1252
Defaulting to user installation because normal site-packages is not writeable


In [3]:
import pandas as pd
import numpy as np
import librosa
import soundfile as sf
import requests
import os
import io
from tqdm import tqdm

INPUT_CSV   = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\disfluency_output.csv"
OUTPUT_CSV  = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\Speech_Disfluencies_Result.csv"
CLIPPED_DIR = r"C:\Users\satya\Downloads\JoshTalks_ASR_Assignment\task\Q2_Disfluency_Detection\clipped_audio"

os.makedirs(CLIPPED_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)
print("Total segments to clip:", len(df))

Total segments to clip: 1690


In [4]:
def download_and_clip(audio_url, start_sec, end_sec, out_path, sr=16000):
    try:
        response = requests.get(audio_url, timeout=30)
        if response.status_code != 200:
            return False

        audio_bytes  = io.BytesIO(response.content)
        y, orig_sr   = librosa.load(audio_bytes, sr=sr, mono=True)

        start_sample = max(0, int(start_sec * sr))
        end_sample   = min(len(y), int(end_sec * sr))

        if start_sample >= end_sample:
            return False

        sf.write(out_path, y[start_sample:end_sample], sr)
        return True

    except Exception as e:
        print(f"Error: {e}")
        return False

In [6]:
saved_paths = []

for idx, row in tqdm(df.head(100).iterrows(), total=100):
    recording_id = str(row['recording_id'])
    start_sec    = float(row['segment_start'])
    end_sec      = float(row['segment_end'])
    dtype        = str(row['disfluency_type'])
    audio_url    = str(row['audio_url'])

    filename = f"{recording_id}_{dtype}_{int(start_sec*1000)}_{int(end_sec*1000)}.wav"
    out_path = os.path.join(CLIPPED_DIR, filename)

    if os.path.exists(out_path):
        saved_paths.append(out_path)
        continue

    success = download_and_clip(audio_url, start_sec, end_sec, out_path)
    saved_paths.append(out_path if success else "")

# Fill remaining rows with empty path
remaining = len(df) - 100
saved_paths.extend([""] * remaining)

df['audio_segment_path'] = saved_paths
print(f"Clips saved: {sum(1 for p in saved_paths if p)}")

100%|██████████| 100/100 [00:00<00:00, 525.15it/s]

Clips saved: 100


In [7]:
final_df = df[[
    'recording_id', 'user_id', 'segment_start', 'segment_end',
    'disfluency_type', 'transcription_snippet', 'audio_segment_path'
]]

final_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')

print("="*45)
print("Speech_Disfluencies_Result.csv saved!")
print("Total rows  :", len(final_df))
print("Clips saved :", final_df['audio_segment_path'].ne('').sum())
print("By type:")
print(final_df['disfluency_type'].value_counts().to_string())
print("="*45)

Speech_Disfluencies_Result.csv saved!
Total rows  : 1690
Clips saved : 100
By type:
disfluency_type
filler          980
repetition      701
prolongation      9


In [12]:
df = pd.read_csv(DISFLUENCY_CSV)

print("Total segments to clip:", len(df))
df.head()

Total segments to clip: 3995


,recording_id,start_time,end_time,disfluency_word,text
0,1020918,4.34,18.65,हूं,जो टाइम था वो गोल्डन टाइम होता है स्कूल के दिन...
1,1020918,19.01,28.40,हां,वहां तो जिम्मेदारियां भी नहीं होती हैं और कम क...
2,1020918,29.90,44.30,हां,की नहीं आपको ये समान लेके आना है कि नहीं आपको ...
3,1020918,66.11,74.93,हां,जी हां
4,1020918,133.25,148.01,तो,जी सर बिल्कुल बहुत ऐसी तो हमारी शरारतें तो बहु...


In [17]:
for idx, row in tqdm(df.iterrows(), total=len(df)):

    recording_id = str(row["recording_id"])
    start = row["start_time"]
    end = row["end_time"]

    audio_path = os.path.join(AUDIO_DIR, recording_id + "_audio.wav")

    if not os.path.exists(audio_path):
        continue

    y, sr = librosa.load(audio_path, sr=16000)

    start_sample = int(start * sr)
    end_sample = int(end * sr)

    clip = y[start_sample:end_sample]

    output_path = os.path.join(
        OUTPUT_DIR,
        f"{recording_id}_{idx}.wav"
    )

    sf.write(output_path, clip, sr)

  0%|          | 0/3995 [00:00<?, ?it/s]

100%|██████████| 3995/3995 [00:00<00:00, 6901.93it/s]


In [18]:
df.head(10)

,recording_id,start_time,end_time,disfluency_word,text
0,1020918,4.34,18.65,हूं,जो टाइम था वो गोल्डन टाइम होता है स्कूल के दिन...
1,1020918,19.01,28.40,हां,वहां तो जिम्मेदारियां भी नहीं होती हैं और कम क...
2,1020918,29.90,44.30,हां,की नहीं आपको ये समान लेके आना है कि नहीं आपको ...
3,1020918,66.11,74.93,हां,जी हां
4,1020918,133.25,148.01,तो,जी सर बिल्कुल बहुत ऐसी तो हमारी शरारतें तो बहु...
5,1020918,148.10,162.26,तो,एक होता है ना थोडे एजेड है वो थोड़ा स्लो स्लो ...
6,1020918,162.26,177.20,अ,देख लो आपको आज तो कुछ समझ में नहीं आ रहा कि क्...
7,1020918,177.26,191.51,अ,हम क्लास नहीं लेंगे सर पढ़ा रहे हैं टीचर आ रहे...
8,1020918,191.60,205.55,अ,टैरिस पे छत पे चले जाते तो सर को गुस्सा आ जाता...
9,1020918,206.18,215.03,अ,अब जो पढ़ाते हैं अरे ऐसे कैसे समझ में नहीं आएग...


In [19]:
df.iterrows()

<generator object DataFrame.iterrows at 0x00000226B48F1850>